# Inspect Jantzen records side by side

Select a paper below. The notebook reads the saved per-paper prediction CSV and the original Jantzen ground-truth CSV, then pairs records with the same greedy match as the evaluation pipeline. It does not call the model or change any results.

`FIRST_RECORD` and `NUMBER_OF_RECORDS` choose matched pairs (1-based, in ground-truth order). Set `RECORD_NUMBERS` to a list such as `[1, 4]` to pick specific pairs. Each heading shows the file positions that were paired. The last cell lists ground-truth rows that were not matched, and prediction rows that matched nothing.


In [6]:
import csv
import sys
from html import escape
from pathlib import Path

from IPython.display import HTML, display

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project directory.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.eval.jantzen import GT_COLUMNS, match_records


In [7]:
PAPER_ID = 107
FIRST_RECORD = 1
NUMBER_OF_RECORDS = 3
RECORD_NUMBERS = None  # Example: [1, 4]; overrides FIRST_RECORD and NUMBER_OF_RECORDS. Indexes matched pairs.

PREDICTION_DIR = PROJECT_ROOT / 'outputs/zeroshot/jantzen_qwen36_27b_fp8_rules_v2'
GROUND_TRUTH_CSV = PROJECT_ROOT / 'data/source/Jantzen_species_interaction/interaction_data.csv'


In [8]:
paper_id = int(PAPER_ID)
if paper_id < 1:
    raise ValueError('PAPER_ID must be a positive number.')
if not GROUND_TRUTH_CSV.is_file():
    raise FileNotFoundError(GROUND_TRUTH_CSV)

with GROUND_TRUTH_CSV.open(encoding='utf-8-sig', newline='') as handle:
    ground_truth_rows = [
        row for row in csv.DictReader(handle) if row['Paper ID'].strip() == str(paper_id)
    ]

prediction_files = sorted((PREDICTION_DIR / 'by_paper').glob(f'paper_{paper_id}__*.csv'))
if len(prediction_files) > 1:
    raise ValueError(f'Multiple prediction files for paper {paper_id}: {prediction_files}')
if prediction_files:
    with prediction_files[0].open(encoding='utf-8', newline='') as handle:
        prediction_rows = list(csv.DictReader(handle))
    prediction_rows.sort(key=lambda row: int(row['record_index']))
else:
    prediction_rows = []

ground_truth_for_match = [
    {field: row.get(column, '') for field, column in GT_COLUMNS.items()}
    for row in ground_truth_rows
]
matches = sorted(
    match_records(ground_truth_for_match, prediction_rows),
    key=lambda match: (match.ground_truth_index, match.prediction_index),
)

print(f'Paper {paper_id}: {len(prediction_rows)} prediction records; '
      f'{len(ground_truth_rows)} ground-truth records')
print(f'Matched {len(matches)}; '
      f'{len(ground_truth_rows) - len(matches)} ground-truth rows missing; '
      f'{len(prediction_rows) - len(matches)} prediction hallucinations')
print(f'Prediction: {prediction_files[0] if prediction_files else "no saved result"}')
print(f'Ground truth: {GROUND_TRUTH_CSV}')


Paper 107: 13 prediction records; 11 ground-truth records
Matched 0; 11 ground-truth rows missing; 13 prediction hallucinations
Prediction: /home/com3dian/Github/Data-extraction-post-training/outputs/zeroshot/jantzen_qwen36_27b_fp8_rules_v2/by_paper/paper_107__0d6b17db.csv
Ground truth: /home/com3dian/Github/Data-extraction-post-training/data/source/Jantzen_species_interaction/interaction_data.csv


In [9]:
FIELDS = [
    ('Paper ID', 'paper_id', 'Paper ID'),
    ('Paper path / DOI', 'paper_path', 'Reference DOI'),
    ('Evidence type', 'evidence_type', None),
    ('Source taxon', 'source_taxon_name', 'sourceTaxonName'),
    ('Interaction type', 'interaction_type_name', 'interactionTypeName'),
    ('Target taxon', 'target_taxon_name', 'targetTaxonName'),
    ('Part of target', 'part_of_target_taxon', 'PartOfTargetTaxon'),
    ('Time of occurrence', 'time_of_occurrence', 'timeOfOccurrence'),
    ('Locality', 'locality_name', 'localityName'),
    ('Latitude', 'decimal_latitude', 'decimalLatitude'),
    ('Longitude', 'decimal_longitude', 'decimalLongitude'),
    ('Observation date', 'observation_date_time', 'observationDateTime'),
    ('Remarks', 'remarks', 'Remarks'),
]


def value(row, field):
    if row is None or field is None:
        return '—'
    content = str(row.get(field) or '').strip()
    return '—' if content.upper() in {'', 'NA'} else escape(content)


if RECORD_NUMBERS is None:
    if FIRST_RECORD < 1 or NUMBER_OF_RECORDS < 1:
        raise ValueError('FIRST_RECORD and NUMBER_OF_RECORDS must be positive.')
    selected_records = list(range(FIRST_RECORD, FIRST_RECORD + NUMBER_OF_RECORDS))
else:
    selected_records = [int(number) for number in RECORD_NUMBERS]
    if not selected_records or any(number < 1 for number in selected_records):
        raise ValueError('RECORD_NUMBERS must contain positive record numbers.')

if not matches:
    print('No matched pairs for this paper.')

for number in selected_records:
    if number > len(matches):
        print(f'Match {number}: this paper has {len(matches)} matched pairs')
        continue
    match = matches[number - 1]
    prediction = prediction_rows[match.prediction_index]
    ground_truth = ground_truth_rows[match.ground_truth_index]
    prediction_number = prediction.get('record_index') or match.prediction_index + 1
    ground_truth_number = match.ground_truth_index + 1
    reversal = '; direction reversed before scoring' if match.oriented else ''

    body = ''.join(
        '<tr>'
        f'<th style="text-align:left;vertical-align:top;padding:5px;border:1px solid #999">{escape(label)}</th>'
        f'<td style="vertical-align:top;padding:5px;border:1px solid #999;overflow-wrap:anywhere">{value(prediction, pred_field)}</td>'
        f'<td style="vertical-align:top;padding:5px;border:1px solid #999;overflow-wrap:anywhere">{value(ground_truth, gt_field)}</td>'
        '</tr>'
        for label, pred_field, gt_field in FIELDS
    )
    display(HTML(
        f'<h3>Match {number} of {len(matches)} — '
        f'prediction {prediction_number}, ground truth {ground_truth_number}, '
        f'identity {match.identity:.3f}{reversal}</h3>'
        '<table style="border-collapse:collapse;width:100%;table-layout:fixed">'
        '<colgroup><col style="width:18%"><col style="width:41%"><col style="width:41%"></colgroup>'
        '<thead><tr><th>Field</th><th>Prediction</th><th>Ground truth</th></tr></thead>'
        f'<tbody>{body}</tbody></table>'
    ))


No matched pairs for this paper.
Match 1: this paper has 0 matched pairs
Match 2: this paper has 0 matched pairs
Match 3: this paper has 0 matched pairs


In [10]:
matched_ground_truth = {match.ground_truth_index for match in matches}
matched_predictions = {match.prediction_index for match in matches}
missing_ground_truth = [
    (index, row)
    for index, row in enumerate(ground_truth_rows, start=1)
    if index - 1 not in matched_ground_truth
]
hallucinations = [
    (index, row)
    for index, row in enumerate(prediction_rows, start=1)
    if index - 1 not in matched_predictions
]


def show_unmatched(title, rows, side):
    print(f'{title}: {len(rows)}')
    for number, row in rows:
        body = ''.join(
            '<tr>'
            f'<th style="text-align:left;vertical-align:top;padding:5px;border:1px solid #999">{escape(label)}</th>'
            f'<td style="vertical-align:top;padding:5px;border:1px solid #999;overflow-wrap:anywhere">'
            f'{value(row, pred_field if side == "prediction" else gt_field)}</td>'
            '</tr>'
            for label, pred_field, gt_field in FIELDS
        )
        label = 'Prediction record' if side == 'prediction' else 'Ground-truth row'
        display(HTML(
            f'<h3>{escape(title)} — {label} {number}</h3>'
            '<table style="border-collapse:collapse;width:100%;table-layout:fixed">'
            '<colgroup><col style="width:22%"><col style="width:78%"></colgroup>'
            f'<tbody>{body}</tbody></table>'
        ))


show_unmatched('Missing ground-truth records', missing_ground_truth, 'ground_truth')
show_unmatched('Hallucinated predictions', hallucinations, 'prediction')


Missing ground-truth records: 11


Paper ID,107
Paper path / DOI,10.1002/ece3.4608
Evidence type,—
Source taxon,Martes martes
Interaction type,eats
Target taxon,Phylloscopus sibillatrix
Part of target,eggs
Time of occurrence,—
Locality,Switzerland
Latitude,—
Longitude,—


Paper ID,107
Paper path / DOI,10.1002/ece3.4608
Evidence type,—
Source taxon,Sciurus vulgaris
Interaction type,eats
Target taxon,Phylloscopus sibillatrix
Part of target,eggs
Time of occurrence,—
Locality,Switzerland
Latitude,—
Longitude,—


Paper ID,107
Paper path / DOI,10.1002/ece3.4608
Evidence type,—
Source taxon,Garrulus glandarius
Interaction type,eats
Target taxon,Phylloscopus sibillatrix
Part of target,eggs
Time of occurrence,—
Locality,Switzerland
Latitude,—
Longitude,—


Paper ID,107
Paper path / DOI,10.1002/ece3.4608
Evidence type,—
Source taxon,Vulpes vulpes
Interaction type,eats
Target taxon,Phylloscopus sibillatrix
Part of target,eggs
Time of occurrence,—
Locality,Switzerland
Latitude,—
Longitude,—


...

Paper ID,paper_107__0d6b17db
Paper path / DOI,/home/com3dian/Github/Data-extraction-post-training/data/ready/jantzen_species_interaction/paper_107/hybrid_auto/paper_107.md
Evidence type,direct observation
Source taxon,Felis catus
Interaction type,preyed upon by
Target taxon,Phylloscopus sibilatrix
Part of target,nest
Time of occurrence,breeding season
Locality,Switzerland
Latitude,—
Longitude,—


Paper ID,paper_107__0d6b17db
Paper path / DOI,/home/com3dian/Github/Data-extraction-post-training/data/ready/jantzen_species_interaction/paper_107/hybrid_auto/paper_107.md
Evidence type,direct observation
Source taxon,Pernis apivorus
Interaction type,preyed upon by
Target taxon,Phylloscopus sibilatrix
Part of target,nest
Time of occurrence,breeding season
Locality,Switzerland
Latitude,—
Longitude,—


Paper ID,paper_107__0d6b17db
Paper path / DOI,/home/com3dian/Github/Data-extraction-post-training/data/ready/jantzen_species_interaction/paper_107/hybrid_auto/paper_107.md
Evidence type,direct observation
Source taxon,Sus scrofa
Interaction type,preyed upon by
Target taxon,Phylloscopus sibilatrix
Part of target,nest
Time of occurrence,breeding season
Locality,Switzerland
Latitude,—
Longitude,—


Paper ID,paper_107__0d6b17db
Paper path / DOI,/home/com3dian/Github/Data-extraction-post-training/data/ready/jantzen_species_interaction/paper_107/hybrid_auto/paper_107.md
Evidence type,direct observation
Source taxon,Sciurus vulgaris
Interaction type,preyed upon by
Target taxon,Phylloscopus sibilatrix
Part of target,nest
Time of occurrence,breeding season
Locality,Switzerland
Latitude,—
Longitude,—


Paper ID,paper_107__0d6b17db
Paper path / DOI,/home/com3dian/Github/Data-extraction-post-training/data/ready/jantzen_species_interaction/paper_107/hybrid_auto/paper_107.md
Evidence type,direct observation
Source taxon,Apodemus
Interaction type,preyed upon by
Target taxon,Phylloscopus sibilatrix
Part of target,nest
Time of occurrence,breeding season
Locality,Switzerland
Latitude,—
Longitude,—
